# 07 · Metals

A metal is the same `embed`, given an arrangement. `rx.isomers(source, geometry=...)` enumerates the distinct
coordination isomers; pick one and embed it. Examples use ionic dative SMILES (donor arrows point to the
metal, anionic donors carry formal charges); `rx.isomers` also accepts neutral/covalent forms such as
`N->[Pt](Cl)Cl`, preferring the written form `N->[Pt+2](<-[Cl-])<-[Cl-]`.

The force field uses a bond-less metal surrogate for typing; the public graph keeps the real metal and
dative bonds.

In [ ]:
import rxembed as rx
import xyzrender
from rdkit import Chem, RDLogger
from rdkit.Chem import rdDistGeom
from rdkit.Chem import rdMolTransforms as T

RDLogger.DisableLog("rdApp.*")
rx.set_verbose("INFO")

## Coordination isomers: the permutations

Square-planar MA₂B₂ gives cis and trans; octahedral MA₃B₃ gives mer and fac, and the same call covers linear,
tetrahedral, trigonal-bipyramidal and square-pyramidal shells. `.summary()` prints each per-vertex arrangement and
Λ/Δ label. Pick one with `.select(index=...)`, `.select(arrangement=...)`, or `.select(hand=...)`. Arrangement is
unambiguous when several isomers share a coarse label.

In [ ]:
for g, smi in {
    "linear": "[Cl-]->[Au+]<-[Cl-]",
    "tetrahedral": "[Cl-]->[Zn+2](<-[Cl-])(<-[Cl-])<-[Cl-]",
    "square_planar": "CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC",
    "octahedral": "N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]",
}.items():
    print(f"{g}:")
    print(rx.isomers(smi, geometry=g).summary())

`rx.isomers` screens out arrangements the ligand backbone cannot reach. This bipyridyl chelate keeps only the
cis assignment; `screen=False` keeps the trans one too, and `summary(details=True)` lists each metal's trans pairs.

In [ ]:
bipy = "[Cl-]->[Ni+2]1(<-[Br-])<-[n]2ccccc2-c2cccc[n]->12"
screened = rx.isomers(bipy, geometry="SPL")
unscreened = rx.isomers(bipy, geometry="SPL", screen=False)
print(f"screened: {len(screened)} isomer(s) | unscreened: {len(unscreened)} isomer(s)")
print(unscreened.summary(details=True))

## Embed one isomer

The fac Co isomer, its coordination sphere drawn as dotted dative bonds, and the trans Pd isomer: the two Cl sit about
180° apart across the metal.

In [ ]:
co = rx.isomers("N->[Co+3](<-N)(<-N)(<-[Cl-])(<-[Cl-])<-[Cl-]", geometry="octahedral").select(
    index=1
)  # the fac isomer (index 1 in .summary())
ens = rx.embed(co, n=1, seed=42)
for report in ens.check().values():
    print(report.summary())
display(xyzrender.render(ens.mol, no_hy=True, idx="s"))

trans = rx.isomers("CCCN->[Pd+2](<-[Cl-])(<-[Cl-])<-NCCC", geometry="square_planar").select(index=1)  # the trans isomer
ens = rx.embed(trans, n=4, seed=42)
site = rx.identity(trans).centres[0]
donors = [v.atoms[0] for v in site.vertices if v and len(v.atoms) == 1]
cl = [d for d in donors if ens.mol.GetAtomWithIdx(d).GetSymbol() == "Cl"]
print(f"{len(ens)} conformers | Cl-Pd-Cl = {ens.measure((cl[0], site.atom, cl[1]))['mean']:.0f}° (trans)")
xyzrender.render(
    ens.mol, no_hy=True, labels=[f"{cl[0] + 1} {site.atom + 1} {cl[1] + 1} a"]
)  # label the trans Cl-Pd-Cl angle

## Optional UFF cleanup trajectory

Capture one cleanup only when requested: the seed plus retained restrained-UFF snapshots on the public graph.

In [ ]:
iso = rx.isomers("C[P]1(C)CC[P](C)(C)->[Ni+2]<-12<-[O-]C(=O)C[N-]->2C", geometry="SPL")[0]
walk = rx.embed(iso, n=1, seed=19, trajectory=True)
cleanup = walk.trajectory  # RDKit Mol; seed followed by cleanup frames
walk.dump_trajectory("cleanup.xyz")
print(cleanup.GetNumConformers(), "frames")

## Metal-centre chirality

rxembed names Λ and Δ independently of atom order. Each requested hand remains separate throughout embedding.

In [ ]:
isos = rx.isomers("[Cl-]->[Co+2]12(<-[Cl-])(<-NCCN->1)<-NCCN->2", geometry="octahedral")  # [Co(en)2Cl2]; bis-chelate + 2 Cl
print(isos.summary())  # trans | achiral ;  cis | Δ ;  cis | Λ
delta = isos.select(hand="Δ")  # pick the Δ enantiomer by its handedness (name-agnostic, unambiguous)
ens = rx.embed(delta, n=8, seed=42).minimize().prune()
reps = ens.representatives()
print(f"cis-Δ [Co(en)2Cl2]: {ens.n} conformers -> {reps.n} distinct mode(s)")
delta_site = rx.identity(delta).centres[0]
for report in reps.check(donors=[v.atoms[0] for v in delta_site.vertices if v and len(v.atoms) == 1]).values():
    print(report.summary())  # includes folded-donor checks
xyzrender.render(reps.mol, no_hy=True, idx="s")

## Chelating ligands: bidentate

A chelate's donors sit on one ligand fragment; rxembed widens their L–M–L window for a rigid backbone bite,
while still separating cis from trans. Ethylenediamine on Pd (5-membered ring, N–Pd–N bite ~82°) with a chloride
and a bromide:

In [ ]:
en = rx.isomers("[Cl-]->[Pd+2]1(<-[Br-])<-NCCN->1", geometry="square_planar")  # the NCCN ring pins both N to Pd
iso = en.select()
site = rx.identity(iso).centres[0]
ens = rx.embed(iso, n=4, seed=42).minimize().prune()
for report in ens.check().values():
    print(report.summary())
nn = [d.atoms[0] for d in site.vertices if d and ens.mol.GetAtomWithIdx(d.atoms[0]).GetSymbol() == "N"]
print(f"{len(ens)} conformers | en bite N-Pd-N = {ens.measure((nn[0], site.atom, nn[1]))['mean']:.0f}°")
xyzrender.render(ens.mol, no_hy=True)

### Two short bites around a tetrahedral centre

The surrounding angle preferences accommodate two disjoint bites together, including pairs joined
by a longer macrocyclic backbone. This does not certify whole-ligand feasibility: ligand bonds and
the selected metal hand still need checking after relaxation.

In [ ]:
smiles = [
    "C=[N]1Cc2cccc[n]2->[Cu+]<-12<-[N](Cc1cccc[n]->21)=C",
    "C1CC[S]2->[Cd+2]34<-[S](C1)CC[S]->3CCCC[S]->4CC2",
]
for smi in smiles:
    isos = rx.isomers(smi, geometry="tetrahedral")
    print(f"{smi}: {len(isos)} isomer(s)")
    for iso in isos:
        ens = rx.embed(iso, n=1, seed=42, threads=1)
        print("relaxed:", not ens.unrelaxed, "| CX match:", rx.cxsmiles(ens.mol) == rx.cxsmiles(iso))
        print(rx.identity(iso).arrangements[0])
        display(xyzrender.render(ens.mol, no_hy=True))

### A tridentate ligand with one axial arm

A TBP equatorial bite can deform while a linked axial donor stays perpendicular to its plane; surrounding
angle targets share that deformation rather than forcing the ideal polyhedron. This selects a seating by
donor slots, not enumeration index, and doesn't establish every alternative is feasible.

In [ ]:
tridentate = rx.isomers("[NH2]1CC[NH]2CC[NH2]->[Cd+2](<-[Cl-])(<-[Cl-])<-1<-2", geometry="trigonal_bipyramidal")
# N0 is terminal and axial; central N3 and terminal N6 are equatorial.
iso = next(candidate for candidate in tridentate
           if 0 in (v := [s.atoms[0] for s in rx.identity(candidate).centres[0].vertices])[:2] and {3, 6} <= set(v[2:]))
ens = rx.embed(iso, n=1, seed=42)
print("relaxed:", not ens.unrelaxed)
print(rx.cxsmiles(ens.mol))

xyzrender.render(ens.mol, no_hy=True)

### Two linked bites in a square-pyramidal shell

The same ligand can place its central donor apically, with its terminal donors on opposite sides.
The bite and spectator targets are constructed together. This selected-state example does not certify every enumerated alternative.

In [ ]:
fans = rx.isomers("[NH2]1CC[NH]2CC[NH2]->[Cd+2](<-[Cl-])(<-[Cl-])<-1<-2", geometry="square_pyramidal")
# The one isomer with both Cl trans in the base: central N3 apical, terminal N0/N6 opposite basal slots.
iso = fans.select(label="trans")
ens = rx.embed(iso, n=1, seed=42)
print("relaxed:", not ens.unrelaxed, "| CX match:", rx.cxsmiles(ens.mol) == rx.cxsmiles(iso))
print(rx.cxsmiles(ens.mol))

xyzrender.render(ens.mol, no_hy=True)

## Side-on η² coordination

A directly π-bonded donor pair binds side-on as one haptic site. rxembed holds the donor bond while applying
both metal distances. The Ni example below keeps its coordinated C=C intact.

In [ ]:
iso = rx.isomers(
    "COC(=O)[C]12->[Ni+2]3(<-[O-]C(=O)C(c4ccccc4)[N-]->3c3ccccc3)<-[C]=1(C(=O)OC)C2(C)C(C)(C)C",
    geometry="square_planar",
).select(index=0)
site = rx.identity(iso).centres[0]
pi = next(v.atoms for v in site.vertices if v and len(v.atoms) > 1)  # the haptic face: the side-on C=C
ens = rx.embed(iso, n=8, seed=42).minimize().prune()
ens.filter("geometry")
c = ens.mol.GetConformer(ens.ids[0])
print(
    f"side-on C=C {pi}: {ens.n} confs | Ni-C {T.GetBondLength(c, site.atom, pi[0]):.2f}/"
    f"{T.GetBondLength(c, site.atom, pi[1]):.2f} Å (both bind) | C=C {T.GetBondLength(c, *pi):.2f} Å (held intact)"
)
xyzrender.render(ens.mol, no_hy=True, labels=[f"{site.atom + 1} {pi[0] + 1} d", f"{site.atom + 1} {pi[1] + 1} d"])

## s-cis and s-trans dienes

A bound diene whose central bond can rotate is s-cis or s-trans, and binding makes each an isomer.
(Butadiene)Fe(CO)3 has three: s-cis (`c`) and s-trans with a positive or negative C1-C2-C3-C4 torsion (`P`, `M`).
The face's slot note in the CXSMILES records the class. A geometry enumerates every class, as it does the
arrangement; `observed=True` keeps the measured one.

In [ ]:
diene = rx.isomers("[O+]#[C-]->[Fe]123(<-[C-]#[O+])(<-[C-]#[O+])<-[CH2]=[CH]->1[CH]->2=[CH2]->3", geometry="TET")
print(diene.summary())
s_trans = next(m for m in diene if any(v and v.winding == "P" for v in rx.identity(m).centres[0].vertices))
ens = rx.embed(s_trans, n=1, seed=42)
print("CX match:", rx.cxsmiles(ens.mol) == rx.cxsmiles(s_trans))
print(rx.cxsmiles(ens.mol))
print(len(rx.isomers(ens.mol, geometry="TET")), "classes from the geometry,", len(rx.isomers(ens.mol, geometry="TET", observed=True)), "measured")

xyzrender.render(ens.mol, no_hy=True)

## Two metal centres in one enumeration

A geometry defines its ligand and haptic stereo, so `rx.isomers` retains those states and enumerates both
metals' arrangements by default; name a centre to vary only that one. The bound amine N5 is a stereocentre
only while bound: `stereo={"locked": "racemic"}` gives both hands with every arrangement. CXSMILES records
one selected state, so reloading returns one combination.

In [ ]:
mnh = rx.read_xyz("structures/mnh.xyz", 0)
mnh_isomers = rx.isomers(mnh, observed=False)
print(f"{len(mnh_isomers)} Mn arrangements (3 mer + 6 fac expected); measured Fe, N, and C stereo retained")
print(mnh_isomers.summary())
mer = mnh_isomers.filter(center="Mn", label="mer")
print(f"{len(mer)} meridional arrangements, one per metal hand (Δ, Λ and achiral)")
chosen = mer.select(hand="lambda")
print(mer.filter(hand="lambda").summary(details=True))
print("N5 racemic:", len(rx.isomers(mnh, stereo={"N5": "racemic"})), "arrangements")
print("locked racemic:", len(rx.isomers(mnh, stereo={"locked": "racemic"})), "arrangements")

### Store and embed one selected state

The CXSMILES carries both metal arrangements and the selected ligand stereochemistry.

In [ ]:
written = rx.cxsmiles(Chem.Mol(chosen, True))  # the statement without the input conformer
loaded = rx.isomers(written, center="all")
print(loaded.summary(details=True))
print(f"{len(loaded)} isomer(s) loaded | round-trips to the written CXSMILES:", rx.cxsmiles(loaded[0]) == written)
print("Fe free-stereo isomer count:", len(rx.isomers(written, center="Fe", stereo="free")))
print(written)
ens = rx.embed(loaded[0], n=1, seed=7)
for report in ens.check().values():
    print(report.summary())
xyzrender.render(xyzrender.load(ens.mol, rebuild=True), no_hy=True, stereo=True)

## Screen the connected chelate, not only its donor separation

The opposed pyridyl arrangements below conflict with the shared ligand/coordination distance model, so `rx.isomers` keeps three arrangements. This screen does not prove chemical impossibility.

In [ ]:
chelate_smiles = "[Cl-]->[Ni+2]12(<-[Cl-])(<-[n]3ccccc3CCc3cccc[n]->13)<-[n]3ccccc3CCc3cccc[n]->23"
chelate_isomers = rx.isomers(chelate_smiles, geometry="OCT")
print(f"{len(chelate_isomers)} arrangements")
print(chelate_isomers.summary())
for candidate in chelate_isomers:
    ensemble = rx.embed(candidate, n=1, seed=42)
    print("relaxed:", not ensemble.unrelaxed, "| CX match:", rx.cxsmiles(ensemble.mol) == rx.cxsmiles(candidate))
    display(xyzrender.render(ensemble.mol, no_hy=True))

## Vacant sites: a coordination pocket

Name a geometry with more vertices than donors and the empty vertex is left as a pocket (`·` in the arrangement) for a substrate to bind later. A 3-donor Pd in a square-planar field leaves one site open.

In [ ]:
pocket = rx.isomers("CC#[N]->[Pd+2](<-[Cl-])<-[Cl-]", geometry="square_planar")  # N + 2 Cl = 3 donors, 4 vertices
print(pocket.summary())  # '·' marks the vacant coordination site
iso = pocket.select(index=0)  # the cis pocket (both Cl cis; the '·' vacancy sits opposite the lone Cl)
ens = rx.embed(iso, n=4, seed=42).minimize().prune()
for report in ens.check().values():
    print(report.summary())
xyzrender.render(ens.mol, no_hy=True, idx="s")  # the '·' vacancy sits opposite the lone Cl

### Native RDKit embedding controls

KDG with native all-in-one refinement (no experimental torsions) is the default for every molecule.
`rx.EmbedParams(native=...)` takes any RDKit `EmbedParameters` for the seed model; rxembed adds the metal
bounds and restrained UFF. Use `srETKDGv3()` for ordinary and small-ring torsions. Retain the native object
for retries; do not share it across concurrent runs.

The two-metal Mn and Fe species below, a Mn complex with a bound H2 and a ferrocenyl phosphine written as CX SMILES,
embeds the same way with `KDG()` as RDKit sets it, which keeps the legacy implementation
(`useLegacyImplementation=True`).

In [ ]:
p = rdDistGeom.KDG()
p.useLegacyImplementation = False
native_iso = rx.isomers("N->[Pt+2](<-[Cl-])(<-[Cl-])<-N", geometry="SPL")[0]
native_ensemble = rx.embed(native_iso, n=1, seed=42, threads=1, params=rx.EmbedParams(native=p))
print("CX match:", rx.cxsmiles(native_ensemble.mol) == rx.cxsmiles(native_iso))

In [ ]:
MN_FE = '[Fe+2:1]12345678(<-[c:8]9%10[P:3](->[Mn+:2]%11%12(<-[N-:6]([C@:48]([c:9]->19[c:10]->2([H:11])[c:12]->3([H:13])[c-:14]->4%10[H:15])([H:49])[C:58]([H:59])([H:60])[H:61])[C:50]([H:51])([H:52])[c:53]1[n:7]->%11[c:54]([H:73])[c:55]([H:72])[c:56]([H:78])[c:57]1[H:71])(<-[C-:62]#[O+:4])(<-[C-:63]#[O+:5])<-[H:64][H:65]->%12)([c:26]1[c:27]([H:28])[c:29]([H:30])[c:31]([H:32])[c:33]([H:34])[c:35]1[H:36])[c:37]1[c:38]([H:39])[c:40]([H:41])[c:42]([H:43])[c:44]([H:45])[c:46]1[H:47])<-[c:16]1([H:17])[c:18]->5([H:19])[c:20]->6([H:21])[c:22]->7([H:23])[c-:24]->81[H:25] |atomProp:0.atomNote.LIN:1.atomNote.s0+:2.atomNote.s0:3.atomNote.OCT-lambda:4.atomNote.s2:6.atomNote.s0+:7.atomNote.s0+:9.atomNote.s0+:11.atomNote.s0+:22.atomNote.s5:31.atomNote.s3:33.atomNote.s4:35.atomNote.s1:36.atomNote.s1:59.atomNote.s1:61.atomNote.s1:63.atomNote.s1:65.atomNote.s1:67.atomNote.s1|'

native = rdDistGeom.KDG()
ens = rx.embed(MN_FE, n=1, seed=1, params=rx.EmbedParams(native=native))
print(ens)
xyzrender.render(ens.mol)

### Unmodified metal graph: native DG and UFF

Use KDG + AIO directly on the real-metal graph, without edited bounds. From identical seeds, compare bond
stretching alone, native UFF with vdW off/on, then added distance or angle restraints.

`distance_k` (kcal/mol/A^2) adds soft +/-0.05 A windows around seed M-L lengths; `angle_k` (kcal/mol/deg^2)
biases the requested polyhedron, not seed angles. Neither scales native UFF, and the SP tag guides KDG but
is not a UFF restraint.

Native vdW pair parameters are `(minimum_distance, well_depth)`, not one atomic radius; switching vdW off
removes all such pairs, not just metal ones.

In [ ]:
from itertools import combinations
from math import isclose
from rdkit.Chem import rdForceFieldHelpers as uff
from rdkit.Chem.rdMolTransforms import GetAngleDeg, GetBondLength

native = Chem.AddHs(Chem.MolFromSmiles("CCN->[Pt@SP1+2](<-NCC)(<-[Cl-])<-[Cl-]"))
requested = rx.isomers(native, geometry="SPL").select(label="cis")
requested_site = rx.identity(requested).centres[0]
requested_cx = rx.cxsmiles(requested)
p = rdDistGeom.KDG()
p.useLegacyImplementation = False
p.randomSeed = 42
p.numThreads = 1
p.embedFragmentsSeparately = False
embed_status = rdDistGeom.EmbedMolecule(native, p)
print("native KDG embed status:", embed_status, "(>=0 succeeded)")
print(rx.geom_check.check(native).summary())
print("native KDG matches the requested cis arrangement:", rx.cxsmiles(native) == requested_cx)
print("UFF types the complete graph:", uff.UFFHasAllMoleculeParams(native))
bond_params = {}
for bond in native.GetBonds():
    pair = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    bond_params[pair] = uff.GetUFFBondStretchParams(native, *pair)
missing = [pair for pair, params in bond_params.items() if params is None]
print(f"bonds without native UFF parameters: {len(missing)} of {len(bond_params)}")
ml_bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx())
            for b in native.GetBonds() if b.GetBondType() == Chem.BondType.DATIVE]
lengths = {(i, j): GetBondLength(native.GetConformer(), i, j) for i, j in ml_bonds}
# The isomer's own compiled donor-metal-donor windows are the polyhedron's angle targets.
angle_targets = {key: sum(window) / 2 for key, window in rx.embed(requested, n=1, seed=42).cons.angles.items()
                  if key[1] == requested_site.atom}
print("Native M-L bond parameters:", {pair: bond_params[pair] for pair in ml_bonds})
paths = Chem.GetDistanceMatrix(native)
vdw_pairs = [(requested_site.atom, atom.GetIdx()) for atom in native.GetAtoms()
             if atom.GetAtomicNum() > 1 and paths[requested_site.atom, atom.GetIdx()] >= 3]
print(f"metal vdW pairs exercised: {len(vdw_pairs)}")
print("Native metal vdW (minimum distance, depth):",
      {pair: uff.GetUFFVdWParams(native, *pair) for pair in vdw_pairs})
print("donor-metal-donor | KDG angle | native UFF (coefficient, angle)")
for (i, metal), (j, other) in combinations(ml_bonds, 2):
    if metal == other:
        print((i, metal, j), round(GetAngleDeg(native.GetConformer(), i, metal, j), 1),
              uff.GetUFFAngleBendParams(native, i, metal, j))
print("model | distance k | angle k | status | max M-L change (A) | geometry OK | CX match")
for model, distance_k, angle_k in (("bonds", 0, 0), ("UFF-no-vdW", 0, 0),
                                  ("UFF", 0, 0), ("UFF", 100, 0), ("UFF", 10000, 0),
                                  ("UFF", 0, 0.01), ("UFF", 0, 0.1)):
    trial = Chem.Mol(native)
    if model == "bonds":
        ff = uff.CreateEmptyForceFieldForMol(trial)
        for (i, j), (k, r0) in bond_params.items():
            ff.UFFAddDistanceConstraint(i, j, False, r0, r0, k)
    else:
        ff = uff.UFFGetMoleculeForceField(trial, vdwThresh=0.0 if model == "UFF-no-vdW" else 10.0,
                                         ignoreInterfragInteractions=False)
    if distance_k:
        for (i, j), length in lengths.items():
            ff.UFFAddDistanceConstraint(i, j, False, length - 0.05, length + 0.05, distance_k)
    if angle_k:
        for (i, m, j), angle in angle_targets.items():
            ff.UFFAddAngleConstraint(i, m, j, False, angle, angle, angle_k)
    ff.Initialize()
    if model == "bonds":
        expected_energy = sum(k * (GetBondLength(trial.GetConformer(), *pair) - r0)**2 / 2
                              for pair, (k, r0) in bond_params.items())
        print(f"  bond-only energy check: formula {expected_energy:.6f} vs FF {ff.CalcEnergy():.6f} "
              f"(match: {isclose(ff.CalcEnergy(), expected_energy, rel_tol=1e-10, abs_tol=1e-10)})")
    status = ff.Minimize(maxIts=2000)
    drift = max(abs(GetBondLength(trial.GetConformer(), i, j) - length)
                for (i, j), length in lengths.items())
    report = rx.geom_check.check(trial)
    matches = rx.cxsmiles(trial) == requested_cx
    print(f"{model:10} | {distance_k:10g} | {angle_k:7g} | {status:6d} | {drift:18.3f} | {report.ok!s:11} | {matches}")
    if angle_k == 0.1:
        print(report.summary())
        print("CX match:", matches)

### Controls for our distance-matrix edits

`EmbedParams(coplanar_14=...)` switches the coplanar 1–4 projections; `metal_floor_relief` switches lowering
carbon-surrogate exclusion floors to metal-aware distances. `donor_orientation` and `conjugation` switch the
optional ligand cleanup terms. All default to `True` and persist through replacement searches; a native
`EmbedParams.native` object stays independent.

In [ ]:
from itertools import product

matrix_iso = rx.isomers("[Pt+2](<-[Cl-])(<-[Cl-])(<-n1ccccc1)<-n1ccccc1", geometry="SPL").select(label="cis")

for names in (("coplanar_14", "metal_floor_relief"), ("donor_orientation", "conjugation")):
    print(*names, "| CX match")
    for values in product((True, False), repeat=2):
        params = rx.EmbedParams(**dict(zip(names, values)))
        ens = rx.embed(matrix_iso, n=1, seed=42, threads=1, params=params)
        print(*values, rx.cxsmiles(ens.mol) == rx.cxsmiles(matrix_iso))